# Gauge-aware divergence and curl at every HEALPix scale

This notebook validates `HealPixDivCurl` with analytic gradient and solid-body rotation fields, then applies `HealPixMultiScaleDivCurl` to a masked `HealPixDecomp` pyramid. Input channel zero is eastward velocity `u`; channel one is northward velocity `v`.

In [ ]:
import healpix_geo
import matplotlib.pyplot as plt
import numpy as np
import torch

from healpix_analyse import (
    HealPixDecomp,
    HealPixDivCurl,
    HealPixMultiScaleDivCurl,
)

torch.manual_seed(3)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float64
RADIUS_M = 6_371_008.8
print("device:", device)

## Partial domain with an artificial island

A projected-reference gauge places its singularities far from this regional domain. The missing circular island exercises the same local-mask behaviour expected near a coastline.

In [ ]:
LEVEL = 6
CENTRE = (20.0, 35.0)
OUTER_RADIUS_DEG = 22.0
candidate_ids, _, _ = healpix_geo.nested.cone_coverage(
    CENTRE, OUTER_RADIUS_DEG, LEVEL, ellipsoid="sphere"
)
candidate_ids = np.asarray(candidate_ids, dtype=np.int64)
candidate_lon, candidate_lat = healpix_geo.nested.healpix_to_lonlat(
    candidate_ids.tolist(), LEVEL, ellipsoid="sphere"
)
candidate_lon = np.asarray(candidate_lon)
candidate_lat = np.asarray(candidate_lat)
island = ((candidate_lon - 27.0) / 3.2) ** 2 + ((candidate_lat - 35.0) / 2.5) ** 2 < 1.0
cell_ids = candidate_ids[~island]
lon, lat = healpix_geo.nested.healpix_to_lonlat(
    cell_ids.tolist(), LEVEL, ellipsoid="sphere"
)
lon = np.asarray(lon, dtype=np.float64)
lat = np.asarray(lat, dtype=np.float64)

plt.figure(figsize=(7, 5))
plt.scatter(lon, lat, c=lat, s=8, cmap="viridis")
plt.xlabel("longitude [deg]")
plt.ylabel("latitude [deg]")
plt.title(f"Irregular level-{LEVEL} domain, N={len(cell_ids)}")
plt.colorbar(label="latitude [deg]")
plt.tight_layout()

## Analytic tangent-vector fields

For a constant unit vector `A` and surface normal `n`, the tangent gradient field `U(A-(A.n)n)` has analytic divergence `-2U(A.n)/R` and zero curl. The solid-body rotation `U(A x n)` has zero divergence and curl `2U(A.n)/R`.

In [ ]:
lon_rad = np.deg2rad(lon)
lat_rad = np.deg2rad(lat)
normal = np.stack((
    np.cos(lat_rad) * np.cos(lon_rad),
    np.cos(lat_rad) * np.sin(lon_rad),
    np.sin(lat_rad),
), axis=1)
east = np.stack((-np.sin(lon_rad), np.cos(lon_rad), np.zeros_like(lon_rad)), axis=1)
north = np.stack((
    -np.sin(lat_rad) * np.cos(lon_rad),
    -np.sin(lat_rad) * np.sin(lon_rad),
    np.cos(lat_rad),
), axis=1)
axis = np.array([0.3, -0.4, 0.8660254])
axis /= np.linalg.norm(axis)
speed = 0.5  # m/s
axis_dot_normal = normal @ axis

gradient_vector = speed * (axis[None, :] - axis_dot_normal[:, None] * normal)
rotation_vector = speed * np.cross(axis[None, :], normal)
gradient_uv = np.stack((
    np.sum(gradient_vector * east, axis=1),
    np.sum(gradient_vector * north, axis=1),
), axis=0)
rotation_uv = np.stack((
    np.sum(rotation_vector * east, axis=1),
    np.sum(rotation_vector * north, axis=1),
), axis=0)
truth_gradient = np.stack((-2.0 * speed * axis_dot_normal / RADIUS_M, np.zeros(len(cell_ids))))
truth_rotation = np.stack((np.zeros(len(cell_ids)), 2.0 * speed * axis_dot_normal / RADIUS_M))

## Single-scale convolution test

Two rotated gauges are retained separately. Their mean is compared with the analytic solution away from the outer boundary and artificial island.

In [ ]:
layer = HealPixDivCurl(
    level=LEVEL,
    cell_ids=cell_ids,
    kernel_sz=5,
    sigma_pix=1.3,
    n_gauges=2,
    gauge_type="projected_ref",
    singularity_lonlat=(20.0, -55.0),
    gauge_reduce="none",
    radius_m=RADIUS_M,
    ellipsoid="sphere",
    dtype=dtype,
    device=device,
)
gradient_estimates = layer(torch.as_tensor(gradient_uv, dtype=dtype, device=device))
rotation_estimates = layer(torch.as_tensor(rotation_uv, dtype=dtype, device=device))
gradient_mean = gradient_estimates.mean(dim=0).detach().cpu().numpy()
rotation_mean = rotation_estimates.mean(dim=0).detach().cpu().numpy()

def angular_distance_deg(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(np.deg2rad, (lon1, lat1, lon2, lat2))
    value = np.sin(lat1) * np.sin(lat2) + np.cos(lat1) * np.cos(lat2) * np.cos(lon1 - lon2)
    return np.rad2deg(np.arccos(np.clip(value, -1.0, 1.0)))

distance_from_centre = angular_distance_deg(lon, lat, *CENTRE)
distance_from_island = angular_distance_deg(lon, lat, 27.0, 35.0)
valid = (distance_from_centre < 17.0) & (distance_from_island > 5.0)
signal_scale = speed / RADIUS_M

metrics = {
    "gradient_div_rel_rms": np.sqrt(np.mean((gradient_mean[0, valid] - truth_gradient[0, valid]) ** 2)) / np.sqrt(np.mean(truth_gradient[0, valid] ** 2)),
    "gradient_curl_scaled_rms": np.sqrt(np.mean(gradient_mean[1, valid] ** 2)) / signal_scale,
    "rotation_div_scaled_rms": np.sqrt(np.mean(rotation_mean[0, valid] ** 2)) / signal_scale,
    "rotation_curl_rel_rms": np.sqrt(np.mean((rotation_mean[1, valid] - truth_rotation[1, valid]) ** 2)) / np.sqrt(np.mean(truth_rotation[1, valid] ** 2)),
    "gradient_gauge_dispersion": gradient_estimates.std(dim=0).square().mean().sqrt().item() / signal_scale,
}
print(layer)
print(metrics)
assert np.isfinite(list(metrics.values())).all()
assert metrics["gradient_div_rel_rms"] < 0.35
assert metrics["rotation_curl_rel_rms"] < 0.35

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)
panels = (
    (gradient_mean[0], "estimated gradient-field divergence"),
    (truth_gradient[0], "analytic gradient-field divergence"),
    (rotation_mean[1], "estimated rotation-field curl"),
    (truth_rotation[1], "analytic rotation-field curl"),
)
for ax, (values, title) in zip(axes.ravel(), panels):
    artist = ax.scatter(lon, lat, c=values, s=9, cmap="coolwarm")
    ax.set_title(title)
    ax.set_xlabel("lon")
    ax.set_ylabel("lat")
    fig.colorbar(artist, ax=ax, label="s$^{-1}$")

## Multiscale divergence and curl

The mixed analytic field is decomposed into three details plus one coarse map. A distinct `HealPixConv` derivative layer is built on each native grid.

In [ ]:
velocity = torch.as_tensor(
    gradient_uv + 0.7 * rotation_uv, dtype=dtype, device=device
).unsqueeze(0)
decomp = HealPixDecomp(
    level=LEVEL, cell_ids=cell_ids, Jmax=3, ellipsoid="sphere",
    dtype=dtype, device=device,
)
velocity_pyramid = decomp(velocity)
multiscale = HealPixMultiScaleDivCurl(
    decomp,
    kernel_sz=3,
    n_gauges=2,
    gauge_type="projected_ref",
    singularity_lonlat=(20.0, -55.0),
    gauge_reduce="mean",
    radius_m=RADIUS_M,
)
diagnostics = multiscale(velocity_pyramid)

print("levels :", diagnostics.levels)
print("cells  :", [band.shape[-1] for band in diagnostics])
print("spacing:", [f"{value / 1000:.2f} km" for value in diagnostics.pixel_spacing_m])
assert diagnostics.levels == velocity_pyramid.levels
assert len(diagnostics) == len(velocity_pyramid)
for band, ids in zip(diagnostics, diagnostics.cell_ids):
    assert band.shape[-2] == 2
    assert band.shape[-1] == len(ids)
    assert torch.isfinite(band).all()
spacing_ratio = np.asarray(diagnostics.pixel_spacing_m[1:]) / np.asarray(diagnostics.pixel_spacing_m[:-1])
assert np.allclose(spacing_ratio, 2.0)

## Autograd

Both the multiscale decomposition and the fixed derivative convolutions remain connected to the input velocity.

In [ ]:
learnable = velocity.detach().clone().requires_grad_(True)
learnable_diagnostics = multiscale(decomp(learnable))
loss = sum(band.square().mean() for band in learnable_diagnostics)
loss.backward()
assert learnable.grad is not None
assert torch.isfinite(learnable.grad).all()
print({"loss": loss.item(), "gradient_norm": learnable.grad.norm().item()})

## Interpretation

Each band now describes divergent and rotational motion over a different physical distance. The native stencil distance approximately doubles from one band to the next, while the `1 / pixel_spacing_m` normalisation keeps every result in the same physical unit. Cells close to mask boundaries should be interpreted with additional care because their convolution stencil is incomplete.